## ⚙️ Passo 0: Configuração do Ambiente

In [ ]:
# (Apenas para Colab - Pule se estiver rodando localmente)
# print("Montando Google Drive...")
# from google.colab import drive
# drive.mount('/content/drive')

import os
# (Apenas para Colab - Pule se estiver rodando localmente)
# project_path_on_drive = '/content/drive/My Drive/bias-aware-community-detection-test'
# os.chdir(project_path_on_drive)
project_path_on_drive = '..'
os.chdir(project_path_on_drive)

print(f"Diretório de trabalho: {os.getcwd()}")

print("📦 Instalando e atualizando dependências...")
# (Use !pip3 no Colab, pip no local)
%pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
%pip install --upgrade networkx python-louvain pandas tqdm psutil transformers[torch] matplotlib seaborn tabulate cvxpy ipywidgets -q
print("✅ Dependências instaladas!")

## 📦 Passo 1: Preparar Estrutura de Pastas e Dados

In [ ]:
# --- Ajuste este caminho ---
# Este é o caminho para a pasta que você criou no Google Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/FACTOID_PROJETO' 
# --------------------------

# 1. Criar as pastas de trabalho no Colab
!mkdir -p /content/FACTOID/social_graph_data
!mkdir -p /content/processed_factoid
!mkdir -p /content/src

print("Estrutura de pastas criada.")

# 2. Descompactar o código-fonte (src/)
print("Descompactando código-fonte (src.zip)...")
!unzip -q -o "{DRIVE_PROJECT_PATH}/src.zip" -d /content/src

# 3. Descompactar os dados do grafo social (social_graph_data.zip)
print("Descompactando dados do grafo (social_graph_data.zip)...")
!unzip -q -o "{DRIVE_PROJECT_PATH}/social_graph_data.zip" -d /content/FACTOID/social_graph_data

# 4. Copiar os arquivos de dados grandes
print("Copiando arquivos de dados (gzip e annotated)...")
!cp "{DRIVE_PROJECT_PATH}/reddit_corpus_unbalanced_filtered.gzip" /content/FACTOID/
!cp "{DRIVE_PROJECT_PATH}/fn_domains_annotated" /content/FACTOID/

print("\nPreparação concluída. Todos os arquivos estão prontos.")

## 🐍 Passo 2: Código Principal (Definições e Funções)

In [ ]:
import pandas as pd
import networkx as nx
import community as community_louvain
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from tqdm import tqdm
import pickle
import os
import datetime
import sys

# --- Adicionar o código-fonte descompactado ao path do Python ---
# Isso é crucial para que 'from src.heuristic...' funcione
sys.path.append('/content/src')

try:
    from src.heuristic import EnhancedLouvainWithBias
    from src.reddit_user_dataset import RedditUserDataset, EvaluatedUser  
    from src.fake_news_detection import AnnotatedFakeNewsDetector 
except ImportError as e:
    print(f"Erro de Importação: {e}")
    print("Verifique se 'src.zip' foi descompactado corretamente na Célula 2.")
    print("Arquivos em /content/src:", os.listdir('/content/src'))
    # sys.exit(1) # No Colab, evitamos sys.exit para não matar o kernel

# --- 0. Definição de Caminhos e Parâmetros ---
# (Caminhos atualizados para o ambiente do Colab)
PATH_BASE_DATASET = '/content/FACTOID/reddit_corpus_unbalanced_filtered.gzip'
PATH_SOCIAL_FOLDER = '/content/FACTOID/social_graph_data/'
PATH_ANNOTATED_DOMAINS = '/content/FACTOID/fn_domains_annotated'
PATH_CACHE_GRAFO = '/content/processed_factoid/social_graph.gml'
PATH_CACHE_INPUTS = '/content/processed_factoid/validation_inputs.pkl'

# Mapeamento para converter o ground-truth em scores numéricos
BIAS_STRING_TO_SCORE = {
    "left": -1.0,
    "left-center": -0.5,
    "center": 0.0,
    "right-center": 0.5,
    "right": 1.0,
    "fake": 0.0, 
    "conspiracy": 0.0,
    "default": 0.0
}

def construir_grafo_social(dataset, social_folder):
    """
    PASSO 1: Construir o Grafo Social (G)
    """
    print("Iniciando Passo 1: Construção do Grafo Social (G)...")
    
    print(f"Lendo interações sociais de {social_folder}...")
    dataset.cache_social_graph(social_folder)
    
    timeframe = (datetime.date(2000, 1, 1), datetime.date(2025, 1, 1))
    
    dataset_com_grafo = dataset.load_social_graph_from_cache(timeframe, inplace=False)
    
    G = nx.Graph()
    user_ids = set(dataset.data_frame['user_id'])
    
    for _, row in tqdm(dataset_com_grafo.data_frame.iterrows(), total=len(dataset_com_grafo.data_frame), desc="Construindo G"):
        user_a = row['user_id']
        G.add_node(user_a)
        
        for user_b, weight in row['social_graph'].items():
            if user_b in user_ids:
                G.add_edge(user_a, user_b, weight=weight)
                
    print(f"Grafo Social (G) construído: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas.")
    
    os.makedirs(os.path.dirname(PATH_CACHE_GRAFO), exist_ok=True)
    nx.write_gml(G, PATH_CACHE_GRAFO)
    print(f"Grafo salvo em {PATH_CACHE_GRAFO}")
    return G

def gerar_inputs_de_vies(dataset, domains_file):
    """
    PASSO 2: Gerar C_GT (Ground-Truth) e b(v) (Vetor de Input)
    """
    print("\nIniciando Passo 2: Geração de C_GT e b(v)...")
    
    detector = AnnotatedFakeNewsDetector(domain_file_path=domains_file, label='fake')
    print(f"Domínios de viés carregados. Total: {len(detector.bias_map)}")
    
    ground_truth_map = {}
    bias_vector_map = {}
    
    for _, row in tqdm(dataset.data_frame.iterrows(), total=len(dataset.data_frame), desc="Mapeando Viés de Usuários"):
        user_id = row['user_id']
        user_obj = EvaluatedUser(user_id, "REDDIT")
        user_obj.own_posts = row['documents']
        
        post_annotations = detector.candidate(user_obj, content_index=1)
        
        user_bias_labels = []
        for post_id, annotations in post_annotations.items():
            for (domain, label, bias_list, factuality) in annotations:
                user_bias_labels.extend(bias_list)
        
        if user_bias_labels:
            dominant_bias_str = max(set(user_bias_labels), key=user_bias_labels.count)
        else:
            dominant_bias_str = 'center'

        ground_truth_map[user_id] = dominant_bias_str
        bias_vector_map[user_id] = BIAS_STRING_TO_SCORE.get(dominant_bias_str, BIAS_STRING_TO_SCORE["default"])
            
    print(f"Inputs de viés (C_GT e b(v)) gerados para {len(ground_truth_map)} usuários.")
    
    os.makedirs(os.path.dirname(PATH_CACHE_INPUTS), exist_ok=True)
    with open(PATH_CACHE_INPUTS, 'wb') as f:
        pickle.dump({'gt_map': ground_truth_map, 'b_v_map': bias_vector_map}, f)
    print(f"Inputs salvos em {PATH_CACHE_INPUTS}")
    return ground_truth_map, bias_vector_map

def executar_validacao(G, b_v_map, gt_map):
    """
    PASSO 4: Execução Comparativa (Louvain vs. Heurística)
    """
    print("\nIniciando Passo 4: Execução Comparativa...")
    
    nodes = list(G.nodes())
    labels_true = []
    labels_louvain = []
    labels_heuristica = []
    
    print("Executando Baseline (Louvain Padrão, alpha=0.0)...")
    partition_louvain = community_louvain.best_partition(G, weight='weight', random_state=42)
    
    print("Executando Proposta (Enhanced Louvain, alpha=0.5)...")
    
    enhanced_model = EnhancedLouvainWithBias(
        alpha=0.5, 
        max_iterations=100, 
        verbose=True
    )
    
    enhanced_model.fit(G, b_v_map, num_communities=2)
    partition_heuristica = enhanced_model.get_communities()

    print("\nCalculando métricas de validação...")
    
    valid_nodes = (
        set(nodes) & 
        set(gt_map.keys()) & 
        set(partition_louvain.keys()) & 
        set(partition_heuristica.keys())
    )
    
    if not valid_nodes:
        print("Erro: Nenhum nó em comum encontrado entre o grafo e os mapas de viés.")
        return

    for node in valid_nodes:
        labels_true.append(gt_map[node])
        labels_louvain.append(partition_louvain[node])
        labels_heuristica.append(partition_heuristica[node])
            
    ari_louvain = adjusted_rand_score(labels_true, labels_louvain)
    nmi_louvain = normalized_mutual_info_score(labels_true, labels_louvain)
    
    ari_heuristica = adjusted_rand_score(labels_true, labels_heuristica)
    nmi_heuristica = normalized_mutual_info_score(labels_true, labels_heuristica)
    
    print("\n" + "="*50)
    print("--- RESULTADOS FINAIS DA VALIDAÇÃO (Parte 1) ---")
    print(f"Baseline (Louvain, \u03B1=0.0) | ARI: {ari_louvain:.4f} | NMI: {nmi_louvain:.4f}")
    print(f"Proposta (Heurística, \u03B1=0.5) | ARI: {ari_heuristica:.4f} | NMI: {nmi_heuristica:.4f}")
    print("="*50)
    
    if ari_heuristica > ari_louvain:
        print("\n[SUCESSO] Validação bem-sucedida: A Heurística (alpha=0.5) superou o Louvain padrão.")
    else:
        print("\n[FALHA] Validação falhou: A Heurística não produziu uma partição melhor que o Louvain padrão.")

print("Todas as funções foram definidas.")

## 🏃 Passo 3: Executar a Validação

In [ ]:
# --- Bloco Principal de Execução ---
# (Este é o único código que realmente "roda" o pipeline)

# --- PASSO 1: Carregar/Construir Grafo G ---
G = None
if os.path.exists(PATH_CACHE_GRAFO):
    print(f"Carregando Grafo (G) do cache: {PATH_CACHE_GRAFO}")
    G = nx.read_gml(PATH_CACHE_GRAFO)
else:
    print("Cache do grafo não encontrado. Construindo do zero...")
    base_dataset = RedditUserDataset.load_from_file(PATH_BASE_DATASET, compression='gzip')
    print(f"Dataset base carregado com {len(base_dataset.data_frame)} usuários.")
    G = construir_grafo_social(base_dataset, PATH_SOCIAL_FOLDER)

print(f"Grafo (G) pronto: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas.")

# --- PASSO 2/3: Carregar/Gerar Inputs de Viés ---
gt_map = None
b_v_map = None
if os.path.exists(PATH_CACHE_INPUTS):
    print(f"Carregando inputs de viés (C_GT, b(v)) do cache: {PATH_CACHE_INPUTS}")
    with open(PATH_CACHE_INPUTS, 'rb') as f:
        data = pickle.load(f)
        gt_map = data['gt_map']
        b_v_map = data['b_v_map']
else:
    print("Cache de inputs de viés não encontrado. Construindo do zero...")
    if 'base_dataset' not in locals():
        base_dataset = RedditUserDataset.load_from_file(PATH_BASE_DATASET, compression='gzip')
        print(f"Dataset base carregado com {len(base_dataset.data_frame)} usuários.")
    gt_map, b_v_map = gerar_inputs_de_vies(base_dataset, PATH_ANNOTATED_DOMAINS)

print(f"Inputs de Viés (C_GT e b(v)) prontos.")

# --- PASSO 4: Executar Validação ---
executar_validacao(G, b_v_map, gt_map)